# ScoreWise · EDA (E1/T1.2)
Dados: *Give Me Some Credit*. Todas as figuras são exportadas para `reports/figures/eda_*.png`. A EDA usa o dataset limpo e deduplicado **inteiro**, só para descrever; nenhuma decisão de modelagem aprende parâmetros daqui (imputação e clip ficam no Pipeline).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from config import FIGURES_DIR, RAW_FILE, DELINQUENCY_COLS, AGE_BINS
from data import load_raw, clean_static, deduplicate, file_sha256

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
COR, COR_ALERTA = "#2a6f97", "#c1443c"


def salvar(fig, nome):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / nome, dpi=150)
    plt.show()
    plt.close(fig)

## 1.2.1 Shape, tipos e `describe()`

In [2]:
raw = load_raw()
print("SHA-256:", file_sha256())
print("shape bruto:", raw.shape)
display(raw.dtypes.to_frame("dtype").T)
display(raw.describe().T.round(2))

SHA-256: 1bd46da486a5708c58c7b01a034fae2a13b327f6f7b62ea7ba4fe3b5824b24ac
shape bruto: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
dtype,int64,float64,int64,int64,float64,float64,int64,int64,int64,int64,float64


,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,150000.0,0.07,0.25,0.0,0.00,0.00,0.00,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.05,249.76,0.0,0.03,0.15,0.56,50708.0
age,150000.0,52.30,14.77,0.0,41.00,52.00,63.00,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.42,4.19,0.0,0.00,0.00,0.00,98.0
DebtRatio,150000.0,353.01,2037.82,0.0,0.18,0.37,0.87,329664.0
MonthlyIncome,120269.0,6670.22,14384.67,0.0,3400.00,5400.00,8249.00,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.45,5.15,0.0,5.00,8.00,11.00,58.0
NumberOfTimes90DaysLate,150000.0,0.27,4.17,0.0,0.00,0.00,0.00,98.0
NumberRealEstateLoansOrLines,150000.0,1.02,1.13,0.0,0.00,1.00,2.00,54.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.24,4.16,0.0,0.00,0.00,0.00,98.0


`DebtRatio` tem máximo de 329.664 e `RevolvingUtilizationOfUnsecuredLines` chega a 50.708, valores impossíveis para razões. Isso justifica o *clip* p1–p99.

In [3]:
df_pt = clean_static(raw)
df = deduplicate(df_pt)
print("shape final:", df.shape)

deduplicate: 609 linhas removidas (150000 -> 149391)
shape final: (149391, 12)


## 1.2.2 Nulos (% por coluna)

In [4]:
nulos_raw = (raw.isna().mean() * 100).round(2)
display(nulos_raw[nulos_raw > 0].to_frame("% nulos (bruto)"))
nulos = (df.isna().mean() * 100).round(2)
display(nulos[nulos > 0].to_frame("% nulos (após clean_static + dedup)"))

,% nulos (bruto)
MonthlyIncome,19.82
NumberOfDependents,2.62


,% nulos (após clean_static + dedup)
atrasos_30_59,0.15
renda_mensal,19.56
atrasos_90,0.15
atrasos_60_89,0.15
dependentes,2.56


`MonthlyIncome` tem 19,82% de nulos e `NumberOfDependents`, 2,62%. As demais colunas não têm nulos no bruto. A ausência de renda será mantida como informação (flag `renda_ausente`) e não só imputada.

## 1.2.3 Distribuição do alvo

In [5]:
taxa = df["alvo"].mean()
contagem = df["alvo"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Bom (0)", "Mau (1)"], contagem.values, color=[COR, COR_ALERTA])
for i, v in enumerate(contagem.values):
    ax.text(i, v, f"{v:,}\n({v/len(df):.1%})".replace(",", "."), ha="center", va="bottom")
ax.set_ylabel("Propostas")
ax.set_title("Distribuição do alvo")
ax.set_ylim(0, contagem.max() * 1.15)
salvar(fig, "eda_alvo.png")
print(f"taxa de mau pagador: {taxa:.4%}")

taxa de mau pagador: 6.6999%


C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Só ~6,7% das propostas são de maus pagadores (cerca de 1 em 15). Acurácia seria enganosa (prever sempre "bom" acerta ~93%), por isso usamos AUC/KS, `class_weight` e corte por custo.

## 1.2.4 Taxa de mau por faixa de idade

In [6]:
df["faixa_idade"] = pd.cut(df["idade"], bins=[17, 29, 44, 59, 120], labels=["18–29", "30–44", "45–59", "60+"])
t = df.groupby("faixa_idade", observed=True)["alvo"].agg(["mean", "size"])
display(t.round(4))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(t.index.astype(str), t["mean"] * 100, color=COR)
for i, v in enumerate(t["mean"] * 100):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax.axhline(taxa * 100, ls="--", color="gray", lw=1)
ax.text(3.45, taxa * 100 + .15, "média", color="gray", ha="right")
ax.set_ylabel("% de maus pagadores")
ax.set_title("Inadimplência por faixa etária")
salvar(fig, "eda_idade.png")

,mean,size
faixa_idade,,
18–29,0.1192,8609
30–44,0.0950,38920
45–59,0.0704,53788
60+,0.0312,48073


C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


A inadimplência cai de forma monótona com a idade: ~11,9% até 29 anos contra ~3,1% acima de 59 (quase 4×). Idade é muito preditiva, mas é atributo sensível, o que alimenta a análise de equidade (E5).

## 1.2.5 Taxa de mau por nº de atrasos 90+

In [7]:
a90 = df["atrasos_90"].clip(upper=5)
t = df.groupby(a90)["alvo"].agg(["mean", "size"])
t.index = [str(int(i)) if i < 5 else "5+" for i in t.index]
display(t.round(4))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(t.index, t["mean"] * 100, color=COR)
for i, v in enumerate(t["mean"] * 100):
    ax.text(i, v, f"{v:.0f}%", ha="center", va="bottom")
ax.set_xlabel("Nº de atrasos ≥ 90 dias")
ax.set_ylabel("% de maus pagadores")
ax.set_title("Inadimplência por atrasos 90+")
salvar(fig, "eda_atrasos90.png")

,mean,size
0,0.0464,141108
1,0.3372,5232
2,0.4990,1555
3,0.5772,667
4,0.6701,291
5+,0.6518,313


C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Um único atraso de 90+ dias leva a inadimplência de ~4,6% (sem atraso) para ~33,7%, e com 2 atrasos chega a ~50%. O histórico de atraso é o sinal mais forte do dataset (maior correlação de Spearman com o alvo).

## 1.2.6 Taxa de mau por decil de utilização do limite

In [8]:
util = df["utilizacao_limite"].clip(upper=df["utilizacao_limite"].quantile(0.99))  # só visual
decil = pd.qcut(util, 10, duplicates="drop")
t = df.groupby(decil, observed=True)["alvo"].mean() * 100
rotulos = [f"D{i+1}" for i in range(len(t))]
display(pd.DataFrame({"faixa": t.index.astype(str), "% mau": t.values.round(2)}))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(rotulos, t.values, color=COR)
for i, v in enumerate(t.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Decil de utilização do limite (D10 = mais utilizado)")
ax.set_ylabel("% de maus pagadores")
ax.set_title("Inadimplência por decil de utilização")
salvar(fig, "eda_utilizacao.png")

,faixa,% mau
0,"(-0.001, 0.0032]",2.56
1,"(0.0032, 0.0194]",1.34
2,"(0.0194, 0.0437]",1.41
3,"(0.0437, 0.0834]",1.91
4,"(0.0834, 0.154]",2.38
5,"(0.154, 0.271]",3.49
6,"(0.271, 0.443]",5.21
7,"(0.443, 0.695]",8.78
8,"(0.695, 0.978]",16.41
9,"(0.978, 1.094]",23.51


C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


A relação é não linear: nos 5 primeiros decis fica entre ~1,3% e ~2,6% e dispara nos dois últimos (~16,4% e ~23,5%). Quem usa quase todo o limite é o perfil de maior risco, e isso favorece modelos de árvore sobre a logística.

## 1.2.7 Correlação (Spearman)

In [9]:
num = df.drop(columns=["faixa_idade"])
corr = num.corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
ax.set_title("Correlação de Spearman")
salvar(fig, "eda_correlacao.png")
display(corr["alvo"].drop("alvo").sort_values().round(3).to_frame("Spearman com o alvo"))

C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Spearman com o alvo
idade,-0.117
renda_mensal,-0.067
linhas_abertas,-0.040
financ_imobiliarios,-0.035
razao_divida,0.020
dependentes,0.046
atraso_codigo_especial,0.083
utilizacao_limite,0.241
atrasos_30_59,0.251
atrasos_60_89,0.268


As três colunas de atraso são fortemente correlacionadas entre si e são as que mais se associam ao alvo (Spearman 0,25 a 0,34), seguidas de utilização do limite (0,24); idade é a principal correlação negativa (−0,12). A redundância entre os atrasos justifica a feature `total_atrasos`.

## 1.2.8 Evidências das anomalias

In [10]:
print("age == 0 (bruto):", int((raw["age"] == 0).sum()))
for c in DELINQUENCY_COLS:
    print(f"{c}: 96/98 = {int(raw[c].isin([96, 98]).sum())}")
mask = raw[DELINQUENCY_COLS].isin([96, 98]).any(axis=1)
print("linhas com 96/98 (bruto):", int(mask.sum()), "| valores:", sorted(raw.loc[mask, DELINQUENCY_COLS[0]].unique()))
print("taxa de mau nessas linhas (bruto):", f"{raw.loc[mask, 'SeriousDlqin2yrs'].mean():.1%}", "vs resto:", f"{raw.loc[~mask, 'SeriousDlqin2yrs'].mean():.1%}")
sem_renda = df["renda_mensal"].isna()
print("DebtRatio mediana | renda nula:", round(df.loc[sem_renda, "razao_divida"].median(), 1),
      "| renda informada:", round(df.loc[~sem_renda, "razao_divida"].median(), 2))
print("DebtRatio > 1 | renda nula:", int((df.loc[sem_renda, "razao_divida"] > 1).sum()), "de", int(sem_renda.sum()),
      "| renda informada:", int((df.loc[~sem_renda, "razao_divida"] > 1).sum()), "de", int((~sem_renda).sum()))
print("utilizacao_limite > 1:", int((df["utilizacao_limite"] > 1).sum()), "| máx:", df["utilizacao_limite"].max())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vc = raw["NumberOfTimes90DaysLate"].value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color=COR)
axes[0].set_yscale("log")
axes[0].set_title("Atrasos 90+ (bruto): note 96 e 98")
axes[0].tick_params(axis="x", labelsize=6, rotation=90)
axes[0].set_xlabel("valor")
axes[0].set_ylabel("linhas (log)")
dr = df["razao_divida"].clip(upper=df["razao_divida"].quantile(0.99))
axes[1].hist([dr[~sem_renda], dr[sem_renda]], bins=40, label=["renda informada", "renda nula"], color=[COR, COR_ALERTA], density=True)
axes[1].set_title("DebtRatio (clip p99) por presença de renda")
axes[1].legend()
axes[1].set_xlabel("razao_divida")
axes[2].boxplot(df["utilizacao_limite"].clip(upper=df["utilizacao_limite"].quantile(0.99)))
axes[2].set_title("Utilização do limite (clip p99)")
axes[2].set_xticks([])
salvar(fig, "eda_anomalias.png")

age == 0 (bruto): 1
NumberOfTime30-59DaysPastDueNotWorse: 96/98 = 269
NumberOfTime60-89DaysPastDueNotWorse: 96/98 = 269
NumberOfTimes90DaysLate: 96/98 = 269
linhas com 96/98 (bruto): 269 | valores: [np.int64(96), np.int64(98)]
taxa de mau nessas linhas (bruto): 54.6% vs resto: 6.6%
DebtRatio mediana | renda nula: 1198.0 | renda informada: 0.3
DebtRatio > 1 | renda nula: 27897 de 29221 | renda informada: 7233 de 120170
utilizacao_limite > 1: 3321 | máx: 50708.0


C:\Users\Samuel\AppData\Local\Temp\ipykernel_11732\3386021687.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


(a) Os códigos 96/98 não são contagens: são marcadores especiais, e essas linhas têm taxa de mau de ~54,6% contra ~6,6% no restante, então o sinal é útil e vira a flag `atraso_codigo_especial`. (b) Com renda nula, `DebtRatio` passa a ser a dívida absoluta (mediana ~1.198 contra ~0,30), por isso `renda_ausente` e o clip p1–p99 são necessários. (c) Há `age = 0` (1 linha) e utilização acima de 1, com máximo de 50.708.

## 1.2.9 / 1.2.10 Achados para os slides

1. **Idade:** a inadimplência cai de ~11,9% (até 29 anos) para ~3,1% (60+), portanto há variável sensível e muito preditiva ao mesmo tempo.
2. **Atrasos:** um único atraso de 90+ dias multiplica o risco por ~7× (4,6% → 33,7%); o histórico de atraso é o sinal mais forte.
3. **Utilização do limite:** o risco é baixo até o 5º decil e salta para ~16–24% nos dois últimos, uma relação não linear que favorece o *gradient boosting*.

In [11]:
print(sorted(p.name for p in FIGURES_DIR.glob("eda_*.png")))

['eda_alvo.png', 'eda_anomalias.png', 'eda_atrasos90.png', 'eda_correlacao.png', 'eda_idade.png', 'eda_utilizacao.png']
